In [13]:
from pydantic import BaseModel, Field
from typing import List, Dict, Any
import sys
from pathlib import Path
from langchain.agents import create_agent

# In Jupyter, use current working directory instead of __file__
current_dir = Path.cwd()

# Ensure the repository's 'src' directory is on sys.path so 'workflows' can be imported.
# If we're in 'agents' (src/agents) use its parent (src). If we're directly in 'src' use it.
if current_dir.name == "agents":
    db_dir = current_dir.parent
elif current_dir.name == "src":
    db_dir = current_dir
else:
    # climb up to find a 'src' folder, fallback to current_dir
    p = current_dir
    while p.parent != p and p.name != "src":
        p = p.parent
    db_dir = p if p.name == "src" else current_dir

if str(db_dir) not in sys.path:
    sys.path.insert(0, str(db_dir))

from workflows.state import ResearchState
from llm.geminiAi import model

In [10]:
class plan(BaseModel):
    plan: Dict[str, Any] = Field(
        default_factory=dict,
        description="Generated research plan"
    )

    target_audience: List[str] = Field(
        default_factory=list,
        description="Intended audience for the report"
    )

    selected_agents: List[str] = Field(
        default_factory=list,
        description="Agents selected to perform research tasks"
    )


In [14]:
planner_agent = create_agent(model , response_format=plan ,)

In [ ]:
from typing import cast, Any

def plan_research(query: str) -> plan:
    
    # cast to Any to satisfy the invoke type hint, then normalize response to our pydantic model
    response = planner_agent.invoke(input=cast(Any, query))

    if isinstance(response, plan):
        research_plan = response
    else:
        try:
            research_plan = plan.parse_obj(response)
        except Exception:
            research_plan = plan()
    
    return research_plan